### --------------------------------------------------
## Data Validation PipeLine
### --------------------------------------------------

### Data Validation

##### 1. Import Numpy and Pandas Setup

In [45]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

##### 2. Loading CSV and Defining DataFrame

In [22]:
# -------------------------------------------------
# 2. Load CSV and Clean to Base Financial Data Only
# -------------------------------------------------

df = pd.read_csv("../data/processed/elv_full_metrics.csv")

# Normalize column names
df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
)

print("\nTotal columns detected (before cleaning):", len(df.columns))


# -------------------------------------------------
# Identify Derived Columns (to drop)
# -------------------------------------------------

derived_columns = [
    "profit_margin",
    "profit_margin_in_%",
    "return_on_equity",
    "roe",
    "equity_ratio",
    "equity_ratio_in_%",
    "debt_ratio",
    "cash_liquidity",
    "cash_liquidity_in_%",
    "cagr_21_24",
    "structural_drift",
    "asset_turnover",
    "ebitda_margin",
    "yoy_revenue_growth",
    "return_on_total_capital",
    "return_on_equity_in_%",
    "return_on_total_capital_in_%"
]

# Only drop those that actually exist
cols_to_drop = [col for col in derived_columns if col in df.columns]

if cols_to_drop:
    print("\nDropping derived columns:")
    for col in cols_to_drop:
        print("-", col)
    df = df.drop(columns=cols_to_drop)
else:
    print("\nNo derived columns found.")

print("\nTotal columns after cleaning:", len(df.columns))
print(sorted(df.columns))


Total columns detected (before cleaning): 61

Dropping derived columns:
- profit_margin
- profit_margin_in_%
- return_on_equity
- equity_ratio
- equity_ratio_in_%
- debt_ratio
- cash_liquidity
- cash_liquidity_in_%
- cagr_21_24
- yoy_revenue_growth
- return_on_total_capital
- return_on_equity_in_%
- return_on_total_capital_in_%

Total columns after cleaning: 46
['accounts_payable', 'accounts_receivable', 'cash_register_and_bank', 'company', 'current_assets', 'current_liabilities', 'ebitda', 'effective_firms', 'employees', 'equity', 'financial_cost_ratio', 'financial_costs', 'financial_fixed_assets', 'financial_income', 'fixed_assets', 'free_equity', 'hhi', 'hhi_contribution', 'intangible_fixed_assets', 'inventory', 'liquidity_ratio', 'long-term_liabilities', 'market_share', 'net_income', 'net_margin', 'net_sales', 'operating_expense_ratio', 'operating_expenses', 'operating_margin', 'operating_profit_after_depreciation', 'other_turnover', 'personnel_cost_per_employee', 'personnel_costs

In [23]:
# -------------------------------------------------
# 1. Column-Level NaN Audit
# -------------------------------------------------

nan_summary = (
    df
    .isna()
    .sum()
    .to_frame("nan_count")
)

nan_summary["total_rows"] = len(df)
nan_summary["nan_%"] = (
    nan_summary["nan_count"] / len(df) * 100
).round(2)

nan_summary = nan_summary.sort_values("nan_%", ascending=False)

print("\n--- COLUMN NaN SUMMARY ---")
display(nan_summary)


--- COLUMN NaN SUMMARY ---


,nan_count,total_rows,nan_%
personnel_cost_per_employee,39,40,97.5
net_income,24,40,60.0
results_for_the_year,16,40,40.0
personnel_costs_per_employee_(in_1000),5,40,12.5
employees,3,40,7.5
turnover,0,40,0.0
provisions,0,40,0.0
result_after_net_financial_items,0,40,0.0
stock_change,0,40,0.0
tangible_fixed_assets,0,40,0.0


In [24]:
# -------------------------------------------------
# 2. Identify Fully Broken Columns
# -------------------------------------------------

fully_nan_cols = nan_summary[
    nan_summary["nan_count"] == len(df)
].index.tolist()

print("\nFully NaN columns:", fully_nan_cols)



Fully NaN columns: []


In [25]:
# -------------------------------------------------
# 3.  Critical Financial Columns Audit
# -------------------------------------------------

critical_cols = [
    "turnover",
    "total_assets",
    "equity",
    "results_for_the_year",
    "current_liabilities"
]

critical_nan_rows = df[df[critical_cols].isna().any(axis=1)]
print("\nRows with missing critical financial inputs:")
display(critical_nan_rows[["company", "year"] + critical_cols])
print("\nTotal affected rows:", len(critical_nan_rows))


Rows with missing critical financial inputs:


,company,year,turnover,total_assets,equity,results_for_the_year,current_liabilities
0,Arboga Bilskrot AB,2021,5513.0,1647.0,803.0,NaN,131.0
1,Arboga Bilskrot AB,2022,3831.0,1694.0,808.0,NaN,246.0
2,Arboga Bilskrot AB,2023,3820.0,1447.0,398.0,NaN,256.0
3,Arboga Bilskrot AB,2024,4813.0,1573.0,463.0,NaN,434.0
8,Bilåtervinning i Tibro AB,2021,9247.0,7018.0,4144.0,NaN,1575.0
9,Bilåtervinning i Tibro AB,2022,10150.0,6823.0,4491.0,NaN,1022.0
10,Bilåtervinning i Tibro AB,2023,7449.0,5890.0,4143.0,NaN,1001.0
11,Bilåtervinning i Tibro AB,2024,8328.0,6604.0,4858.0,NaN,1316.0
24,Mariestads Bildemontering och Återvinningstekn...,2021,8030.0,6851.0,4403.0,NaN,1086.0
25,Mariestads Bildemontering och Återvinningstekn...,2022,6996.0,6281.0,4441.0,NaN,914.0



Total affected rows: 16


In [26]:
# Compare missing patterns
mask_net_income_nan = df["net_income"].isna()
mask_results_nan = df["results_for_the_year"].isna()

print("Net income NaNs:", mask_net_income_nan.sum())
print("Results for the year NaNs:", mask_results_nan.sum())

# Overlap analysis
both_nan = (mask_net_income_nan & mask_results_nan).sum()
only_net_income_nan = (mask_net_income_nan & ~mask_results_nan).sum()
only_results_nan = (~mask_net_income_nan & mask_results_nan).sum()

print("\nOverlap diagnostics:")
print("Both NaN:", both_nan)
print("Only net_income NaN:", only_net_income_nan)
print("Only results_for_the_year NaN:", only_results_nan)

Net income NaNs: 24
Results for the year NaNs: 16

Overlap diagnostics:
Both NaN: 0
Only net_income NaN: 24
Only results_for_the_year NaN: 16


In [27]:
df[
    (df["net_income"].notna()) &
    (df["results_for_the_year"].isna())
][["company","year","net_income","results_for_the_year"]]

,company,year,net_income,results_for_the_year
0,Arboga Bilskrot AB,2021,153.0,NaN
1,Arboga Bilskrot AB,2022,5.0,NaN
2,Arboga Bilskrot AB,2023,-410.0,NaN
3,Arboga Bilskrot AB,2024,78.0,NaN
8,Bilåtervinning i Tibro AB,2021,226.0,NaN
9,Bilåtervinning i Tibro AB,2022,347.0,NaN
10,Bilåtervinning i Tibro AB,2023,-349.0,NaN
11,Bilåtervinning i Tibro AB,2024,716.0,NaN
24,Mariestads Bildemontering och Återvinningstekn...,2021,295.0,NaN
25,Mariestads Bildemontering och Återvinningstekn...,2022,38.0,NaN


In [28]:
df[
    (df["net_income"].isna()) &
    (df["results_for_the_year"].notna())
][["company","year","net_income","results_for_the_year"]]

,company,year,net_income,results_for_the_year
4,Autocirc Svensk Bilåtervinning AB,2021,NaN,10681.0
5,Autocirc Svensk Bilåtervinning AB,2022,NaN,856.0
6,Autocirc Svensk Bilåtervinning AB,2023,NaN,22.0
7,Autocirc Svensk Bilåtervinning AB,2024,NaN,1011.0
12,Bjuhrs Bil AB,2021,NaN,650.0
13,Bjuhrs Bil AB,2022,NaN,590.0
14,Bjuhrs Bil AB,2023,NaN,206.0
15,Bjuhrs Bil AB,2024,NaN,109.0
16,Ekholms Bildemontering AB,2021,NaN,477.0
17,Ekholms Bildemontering AB,2022,NaN,671.0


In [29]:
# -------------------------------------------------
# Merge net_income into results_for_the_year
# -------------------------------------------------

df["results_for_the_year"] = (
    df["results_for_the_year"]
    .combine_first(df["net_income"])
)

# Now drop redundant column
df.drop(columns=["net_income"], inplace=True)

print("Remaining NaNs in results_for_the_year:",
      df["results_for_the_year"].isna().sum())

Remaining NaNs in results_for_the_year: 0


In [30]:
# -------------------------------------------------
# 1. Column-Level NaN Audit
# -------------------------------------------------

nan_summary = (
    df
    .isna()
    .sum()
    .to_frame("nan_count")
)

nan_summary["total_rows"] = len(df)
nan_summary["nan_%"] = (
    nan_summary["nan_count"] / len(df) * 100
).round(2)

nan_summary = nan_summary.sort_values("nan_%", ascending=False)

print("\n--- COLUMN NaN SUMMARY ---")
display(nan_summary)


--- COLUMN NaN SUMMARY ---


,nan_count,total_rows,nan_%
personnel_cost_per_employee,39,40,97.5
personnel_costs_per_employee_(in_1000),5,40,12.5
employees,3,40,7.5
turnover,0,40,0.0
provisions,0,40,0.0
result_after_net_financial_items,0,40,0.0
results_for_the_year,0,40,0.0
stock_change,0,40,0.0
tangible_fixed_assets,0,40,0.0
tax_on_the_year's_profit,0,40,0.0


In [31]:
# Detect dubious column names (e.g. "year" vs "Year", "company" vs "Company")
dubious_columns = [col for col in df.columns if col != col.strip().lower().replace(" ", "_")]
if dubious_columns:
    print(f"\nWarning: Detected potentially inconsistent column names: {dubious_columns}")

# Detect duplicate column names
duplicate_columns = df.columns[df.columns.duplicated()].tolist()

if duplicate_columns:
    print("\n Duplicate columns detected:")
    for col in duplicate_columns:
        print(f"- {col}")
else:
    print("\nNo duplicate columns detected.")

# List all column names
print(f"\nTotal columns detected: {len(df.columns)}")
print(sorted(df.columns))


No duplicate columns detected.

Total columns detected: 45
['accounts_payable', 'accounts_receivable', 'cash_register_and_bank', 'company', 'current_assets', 'current_liabilities', 'ebitda', 'effective_firms', 'employees', 'equity', 'financial_cost_ratio', 'financial_costs', 'financial_fixed_assets', 'financial_income', 'fixed_assets', 'free_equity', 'hhi', 'hhi_contribution', 'intangible_fixed_assets', 'inventory', 'liquidity_ratio', 'long-term_liabilities', 'market_share', 'net_margin', 'net_sales', 'operating_expense_ratio', 'operating_expenses', 'operating_margin', 'operating_profit_after_depreciation', 'other_turnover', 'personnel_cost_per_employee', 'personnel_costs_per_employee_(in_1000)', 'profit_before_tax', 'profit_share', 'provisions', 'result_after_net_financial_items', 'results_for_the_year', 'stock_change', 'tangible_fixed_assets', "tax_on_the_year's_profit", 'total_assets', 'total_equity_and_liabilities', 'turnover', 'untaxed_reserves', 'year']


In [33]:
# -------------------------------------------------
# Validate Primitive Financial Columns
# -------------------------------------------------

income_primitives = [
    "turnover",
    "operating_profit_after_depreciation",
    "result_after_net_financial_items",
    "profit_before_tax",
    "tax_on_the_year's_profit",
    "ebitda"
]

balance_primitives = [
    "total_assets",
    "equity",
    "current_assets",
    "current_liabilities",
    "accounts_receivable",
    "cash_register_and_bank"
]

required_columns = income_primitives + balance_primitives

print("\n--- PRIMITIVE COLUMN VALIDATION ---")

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("\nMissing required primitive columns:")
    for col in missing_columns:
        print("-", col)
    raise ValueError("Primitive validation failed. Cannot proceed.")
else:
    print("\nAll required primitive columns exist.")


# -------------------------------------------------
# Check Fully Empty Primitive Columns
# -------------------------------------------------

fully_empty = [
    col for col in required_columns
    if df[col].isna().all()
]

if fully_empty:
    print("\n⚠ Fully empty primitive columns detected:")
    for col in fully_empty:
        print("-", col)
else:
    print("\nNo fully empty primitive columns.")


--- PRIMITIVE COLUMN VALIDATION ---

All required primitive columns exist.

No fully empty primitive columns.


In [34]:
# -------------------------------------------------
# Rebuild Net Income (Deterministic)
# -------------------------------------------------

df["results_for_the_year"] = (
    df["profit_before_tax"] +
    df["tax_on_the_year's_profit"]
)

In [42]:
# -------------------------------------------------
# Validate Net Income Reconstruction Integrity
# -------------------------------------------------

missing_count = df["results_for_the_year"].isna().sum()

assert missing_count == 0, \
    f"Net income reconstruction failed — {missing_count} missing values detected."

In [ ]:
# -------------------------------------------------
# Rebuild Profitability Metrics
# -------------------------------------------------

def safe_divide(numerator, denominator):
    return np.where(denominator != 0, numerator / denominator, np.nan)

# Net Profit Margin (%)
df["profit_margin"] = safe_divide(
    df["results_for_the_year"],
    df["turnover"]
) * 100

# Operating Margin (%)
df["operating_margin"] = safe_divide(
    df["operating_profit_after_depreciation"],
    df["turnover"]
) * 100

# EBITDA Margin (%)
df["ebitda_margin"] = safe_divide(
    df["ebitda"],
    df["turnover"]
) * 100

In [ ]:
# -------------------------------------------------
# Rebuild Capital Structure Metrics
# -------------------------------------------------

# Equity Ratio
df["equity_ratio"] = safe_divide(
    df["equity"],
    df["total_assets"]
)

# Debt Ratio
df["debt_ratio"] = 1 - df["equity_ratio"]

# Return on Equity (%)
df["return_on_equity"] = safe_divide(
    df["results_for_the_year"],
    df["equity"]
) * 100

# Return on Total Capital (%)
df["return_on_total_capital"] = safe_divide(
    df["operating_profit_after_depreciation"],
    df["total_assets"]
) * 100

In [ ]:
# -------------------------------------------------
# Rebuild Liquidity & Efficiency Metrics
# -------------------------------------------------

# Cash Liquidity (Quick-style)
df["cash_liquidity"] = safe_divide(
    df["cash_register_and_bank"] + df["accounts_receivable"],
    df["current_liabilities"]
)

# Asset Turnover
df["asset_turnover"] = safe_divide(
    df["turnover"],
    df["total_assets"]
)

In [ ]:
# -------------------------------------------------
# Rebuild Time-Series Metrics
# -------------------------------------------------

df = df.sort_values(["company", "year"])

In [ ]:
# -------------------------------------------------
# Rebuild Year-over-Year Revenue Growth
# -------------------------------------------------

df["yoy_revenue_growth"] = (
    df.groupby("company")["turnover"]
      .pct_change() * 100
)

In [43]:
# =================================================
# POST-DERIVATION FINANCIAL INTEGRITY AUDIT
# =================================================

print("\n=================================================")
print("POST-DERIVATION FINANCIAL INTEGRITY AUDIT")
print("=================================================\n")

# -------------------------------------------------
# 1️⃣ Column-Level NaN Summary
# -------------------------------------------------

nan_summary = (
    df.isna()
      .sum()
      .to_frame("nan_count")
)

nan_summary["total_rows"] = len(df)
nan_summary["nan_%"] = (
    nan_summary["nan_count"] / len(df) * 100
).round(2)

nan_summary = nan_summary.sort_values("nan_%", ascending=False)

print("\n--- COLUMN NaN SUMMARY ---")
display(nan_summary)


# -------------------------------------------------
# 2️⃣ Derived Metrics NaN Check
# -------------------------------------------------

derived_metrics = [
    "results_for_the_year",
    "profit_margin",
    "operating_margin",
    "ebitda_margin",
    "equity_ratio",
    "debt_ratio",
    "return_on_equity",
    "return_on_total_capital",
    "cash_liquidity",
    "asset_turnover",
    "yoy_revenue_growth"
]

print("\n--- DERIVED METRICS NaN CHECK ---")
display(
    df[derived_metrics]
      .isna()
      .sum()
      .to_frame("nan_count")
)


# -------------------------------------------------
# 3️⃣ YoY Structural Validation
# -------------------------------------------------

print("\n--- YoY STRUCTURAL CHECK ---")

yoy_check = df.groupby("company")["yoy_revenue_growth"] \
              .apply(lambda x: x.isna().sum())

display(yoy_check)

print("\nExpected: 1 NaN per company (first year only)")


# -------------------------------------------------
# 4️⃣ Sanity Bounds Check
# -------------------------------------------------

print("\n--- SANITY CHECKS ---")

print("Equity ratio < 0:", (df["equity_ratio"] < 0).sum())
print("Equity ratio > 1:", (df["equity_ratio"] > 1).sum())

print("Cash liquidity < 0:", (df["cash_liquidity"] < 0).sum())

print("Profit margin < -200%:", (df["profit_margin"] < -200).sum())
print("Profit margin > 200%:", (df["profit_margin"] > 200).sum())


# -------------------------------------------------
# 5️⃣ Accounting Identity Check
# -------------------------------------------------

print("\n--- ACCOUNTING IDENTITY CHECK ---")

identity_diff = (
    df["total_assets"] -
    df["total_equity_and_liabilities"]
).abs().sum()

print("Total absolute difference:", identity_diff)

assert identity_diff == 0, \
    "Accounting identity violation detected!"

print("\n✔ Audit Complete.")


POST-DERIVATION FINANCIAL INTEGRITY AUDIT


--- COLUMN NaN SUMMARY ---


,nan_count,total_rows,nan_%
personnel_cost_per_employee,39,40,97.5
yoy_revenue_growth,10,40,25.0
personnel_costs_per_employee_(in_1000),5,40,12.5
employees,3,40,7.5
market_share,0,40,0.0
total_assets,0,40,0.0
total_equity_and_liabilities,0,40,0.0
turnover,0,40,0.0
untaxed_reserves,0,40,0.0
operating_margin,0,40,0.0



--- DERIVED METRICS NaN CHECK ---


,nan_count
results_for_the_year,0
profit_margin,0
operating_margin,0
ebitda_margin,0
equity_ratio,0
debt_ratio,0
return_on_equity,0
return_on_total_capital,0
cash_liquidity,0
asset_turnover,0



--- YoY STRUCTURAL CHECK ---


company
Arboga Bilskrot AB                                     1
Autocirc Svensk Bilåtervinning AB                      1
Bilåtervinning i Tibro AB                              1
Bjuhrs Bil AB                                          1
Ekholms Bildemontering AB                              1
Köpings Bildemontering AB                              1
Mariestads Bildemontering och Återvinningsteknik AB    1
Nya Högåsens Bildemontering AB                         1
Vingåkers Bilskrot AB                                  1
Örebro Bildemontering AB                               1
Name: yoy_revenue_growth, dtype: int64


Expected: 1 NaN per company (first year only)

--- SANITY CHECKS ---
Equity ratio < 0: 0
Equity ratio > 1: 0
Cash liquidity < 0: 0
Profit margin < -200%: 0
Profit margin > 200%: 0

--- ACCOUNTING IDENTITY CHECK ---
Total absolute difference: 0.0

✔ Audit Complete.


In [48]:
# =============================================================
# ELV FINANCIAL PIPELINE — CLEAN → REBUILD → VALIDATE → LINEAGE
# =============================================================

import pandas as pd
import numpy as np

# -------------------------------------------------------------
# 1. LOAD DATA + NORMALIZE COLUMN NAMES
# -------------------------------------------------------------

df = pd.read_csv("../data/processed/elv_full_metrics.csv")

df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
)

original_columns = set(df.columns)

print("Original column count:", len(original_columns))


# -------------------------------------------------------------
# 2. REBUILD CORE PRIMITIVES (IF NEEDED)
# -------------------------------------------------------------

# Rebuild results_for_the_year deterministically
if {"profit_before_tax", "tax_on_the_year's_profit"}.issubset(df.columns):
    df["results_for_the_year"] = (
        df["profit_before_tax"] +
        df["tax_on_the_year's_profit"]
    )

# Assert no missing results_for_the_year
assert df["results_for_the_year"].isna().sum() == 0


# -------------------------------------------------------------
# 3. REBUILD DERIVED METRICS
# -------------------------------------------------------------

def safe_divide(numerator, denominator):
    return np.where(denominator != 0, numerator / denominator, np.nan)

# Sort for time-series calculations
df = df.sort_values(["company", "year"])

# Profitability
df["profit_margin"] = safe_divide(
    df["results_for_the_year"],
    df["turnover"]
) * 100

df["return_on_equity"] = safe_divide(
    df["results_for_the_year"],
    df["equity"]
) * 100

# Capital structure
df["equity_ratio"] = safe_divide(
    df["equity"],
    df["total_assets"]
)

df["debt_ratio"] = 1 - df["equity_ratio"]

# Liquidity
df["cash_liquidity"] = safe_divide(
    df["cash_register_and_bank"] + df["accounts_receivable"],
    df["current_liabilities"]
)

# Efficiency
df["asset_turnover"] = safe_divide(
    df["turnover"],
    df["total_assets"]
)

# Growth
df["yoy_revenue_growth"] = (
    df.groupby("company")["turnover"]
      .pct_change() * 100
)


# -------------------------------------------------------------
# 4. VALIDATION CHECKS
# -------------------------------------------------------------

print("\n--- STRUCTURAL CHECKS ---")

# Row consistency
assert not df.duplicated(["company", "year"]).any()
print("✔ No duplicate company-year rows")

# Accounting identity
accounting_diff = (
    df["total_assets"] -
    df["total_equity_and_liabilities"]
).abs().sum()

print("Total accounting difference:", accounting_diff)
assert accounting_diff == 0

# Sanity bounds
assert ((df["equity_ratio"] <= 2) | df["equity_ratio"].isna()).all()
assert ((df["profit_margin"] <= 200) | df["profit_margin"].isna()).all()
assert ((df["profit_margin"] >= -200) | df["profit_margin"].isna()).all()

print("✔ Sanity checks passed")

# YoY structural expectation
yoy_counts = df.groupby("company")["yoy_revenue_growth"].apply(lambda x: x.isna().sum())
print("\nExpected 1 NaN per company (first year only):")
print(yoy_counts)


# -------------------------------------------------------------
# 5. COLUMN LINEAGE TRACKING
# -------------------------------------------------------------

final_columns = set(df.columns)

derived_columns = final_columns - original_columns
primitive_columns = final_columns & original_columns

lineage_df = pd.DataFrame({
    "column": list(final_columns)
})

lineage_df["source"] = lineage_df["column"].apply(
    lambda col: "derived" if col in derived_columns else "original"
)

lineage_df = lineage_df.sort_values(["source", "column"])

print("\n--- COLUMN LINEAGE SUMMARY ---")
display(lineage_df)

print("\nOriginal columns:", len(primitive_columns))
print("Derived columns:", len(derived_columns))


# -------------------------------------------------------------
# 6. EXPORT VALIDATED DATASET
# -------------------------------------------------------------

df.to_csv("../data/processed/elv_validated_metrics.csv", index=False)

print("\n✔ Validated dataset exported: elv_validated_metrics.csv")

Original column count: 59

--- STRUCTURAL CHECKS ---
✔ No duplicate company-year rows
Total accounting difference: 0.0


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [46]:
# -------------------------------------------------
# Export Validated Financial Dataset
# -------------------------------------------------

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
output_path = f"../data/processed/elv_financials_validated_{timestamp}.csv"

df.to_csv(output_path, index=False)

print(f"\nValidated datasetexported to: {output_path} with timestamp {timestamp} ")
print("Rows:", len(df))
print("Columns:", len(df.columns))


Validated datasetexported to: ../data/processed/elv_financials_validated_2026-04-19_15-45.csv with timestamp 2026-04-19_15-45 
Rows: 40
Columns: 54


In [ ]:
# Calculate asset turnover ratio
df["asset_turnover"] = df["turnover"] / df["total_assets"]

# Confirm asset turnover calculation
df[["company", "year", "turnover", "total_assets", "asset_turnover"]].head()

,company,year,turnover,total_assets,asset_turnover
0,Arboga Bilskrot AB,2021,"5,513.00","1,647.00",3.35
1,Arboga Bilskrot AB,2022,"3,831.00","1,694.00",2.26
2,Arboga Bilskrot AB,2023,"3,820.00","1,447.00",2.64
3,Arboga Bilskrot AB,2024,"4,813.00","1,573.00",3.06
4,Autocirc Svensk Bilåtervinning AB,2021,"48,649.00","23,962.00",2.03


In [ ]:
# Load Ekholms Bildemontering AB financial dataset
target = "Ekholms Bildemontering AB"
latest_year = df["year"].max()

df_target = df[df["company"] == target].copy()

print("Latest Year:", latest_year)
print("Years Available:", sorted(df_target["year"].unique()))
print("Detected columns for target company:", sorted(df_target.columns))

# -------------------------------------------------
# Preview First 3 Rows (Full Column Values)
# -------------------------------------------------

display(
    df_target.head(5)
        .style
        .set_caption("Target Data Preview (First 5 Rows)")
        .format(precision=2)
)

Latest Year: 2024
Years Available: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Detected columns for target company: ['accounts_payable', 'accounts_receivable', 'asset_turnover', 'cagr_21_24', 'cash_liquidity', 'cash_liquidity_in_%', 'cash_register_and_bank', 'company', 'current_assets', 'current_liabilities', 'debt_ratio', 'debt_ratio', 'ebitda', 'effective_firms', 'employees', 'equity', 'equity_ratio', 'equity_ratio', 'equity_ratio_in_%', 'financial_cost_ratio', 'financial_costs', 'financial_fixed_assets', 'financial_income', 'fixed_assets', 'free_equity', 'hhi', 'hhi_contribution', 'intangible_fixed_assets', 'inventory', 'liquidity_ratio', 'long-term_liabilities', 'market_share', 'net_income', 'net_margin', 'net_sales', 'operating_expense_ratio', 'operating_expenses', 'operating_margin', 'operating_profit_after_depreciation', 'other_turnover', 'personnel_cost_per_employee', 'personnel_costs_per_employee_(in_1000)', 'profit_before_tax', 'profit_margin', 'profit_ma

,company,year,accounts_payable,accounts_receivable,cash_liquidity,cash_liquidity_in_%,cash_register_and_bank,current_assets,current_liabilities,debt_ratio,ebitda,employees,equity,equity_ratio,equity_ratio_in_%,financial_costs,financial_fixed_assets,financial_income,fixed_assets,free_equity,intangible_fixed_assets,inventory,long-term_liabilities,net_income,net_sales,operating_expenses,operating_profit_after_depreciation,other_turnover,personnel_cost_per_employee,personnel_costs_per_employee_(in_1000),profit_before_tax,profit_margin,profit_margin_in_%,provisions,result_after_net_financial_items,results_for_the_year,return_on_equity,return_on_equity_in_%,return_on_total_capital,return_on_total_capital_in_%,stock_change,tangible_fixed_assets,tax_on_the_year's_profit,total_assets,total_equity_and_liabilities,turnover,untaxed_reserves,operating_margin,net_margin,equity_ratio,debt_ratio,liquidity_ratio,operating_expense_ratio,financial_cost_ratio,yoy_revenue_growth,market_share,hhi_contribution,hhi,effective_firms,cagr_21_24,profit_share,asset_turnover
16,Ekholms Bildemontering AB,2021,79.00,58.00,nan,nan,1688.00,2188.00,1222.00,nan,858.00,4.00,819.00,nan,nan,-15.00,60.00,0.00,80.00,699.00,0.00,272.00,0.00,nan,6169.00,-5577.00,851.00,259.00,nan,662.00,610.00,nan,nan,0.00,837.00,477.00,nan,nan,nan,nan,0.00,20.00,-133.00,2267.00,2267.00,6428.00,227.00,0.14,0.14,0.36,0.64,1.79,-0.90,-0.00,nan,0.04,0.00,0.30,3.39,-0.20,0.03,2.84
17,Ekholms Bildemontering AB,2022,110.00,61.00,nan,nan,2207.00,2432.00,1203.00,nan,1153.00,4.00,792.00,nan,nan,-12.00,60.00,0.00,72.00,672.00,0.00,0.00,0.00,nan,6256.00,-5373.00,1146.00,263.00,nan,692.00,852.00,nan,nan,0.00,1134.00,671.00,nan,nan,nan,nan,0.00,12.00,-181.00,2505.00,2505.00,6519.00,509.00,0.18,0.18,0.32,0.68,2.02,-0.86,-0.00,0.01,0.04,0.00,0.31,3.20,-0.20,0.06,2.60
18,Ekholms Bildemontering AB,2023,102.00,109.00,nan,nan,589.00,1309.00,679.00,nan,161.00,4.00,203.00,nan,nan,-7.00,60.00,5.00,111.00,83.00,0.00,0.00,0.00,nan,3671.00,-4064.00,142.00,535.00,nan,584.00,112.00,nan,nan,0.00,139.00,82.00,nan,nan,nan,nan,0.00,51.00,-30.00,1420.00,1420.00,4206.00,537.00,0.04,0.04,0.14,0.86,1.93,-1.11,-0.00,-0.41,0.02,0.00,0.36,2.81,-0.20,0.01,2.96
19,Ekholms Bildemontering AB,2024,92.00,147.00,nan,nan,629.00,1049.00,587.00,nan,-95.00,4.00,160.00,nan,nan,-26.00,60.00,19.00,95.00,40.00,0.00,0.00,0.00,nan,3214.00,-3561.00,-112.00,233.00,nan,485.00,20.00,nan,nan,0.00,-120.00,7.00,nan,nan,nan,nan,0.00,35.00,-13.00,1144.00,1144.00,3447.00,397.00,-0.03,-0.04,0.14,0.86,1.79,-1.11,-0.01,-0.12,0.02,0.00,0.37,2.71,-0.20,-0.00,3.01


##### 3. Classification Functions

In [181]:

def classify_metric(
    target_value,
    industry_value,
    mode="absolute",        # "absolute", "ratio", "scale"
    threshold=5,
    scale_threshold=25
):

    if pd.isna(target_value) or pd.isna(industry_value):
        return "Undefined"

    target_value = float(target_value)
    industry_value = float(industry_value)

    if mode == "scale" and industry_value == 0:
        return "Undefined"

    if mode in ["absolute", "ratio"]:
        gap = target_value - industry_value

        if gap >= threshold:
            return "Strong"
        elif gap <= -threshold:
            return "Weak"
        else:
            return "Neutral"

    elif mode == "scale":
        relative = (target_value / industry_value - 1) * 100

        if relative >= scale_threshold:
            return "Large"
        elif relative <= -scale_threshold:
            return "Subscale"
        else:
            return "Mid-scale"

    else:
        return "Invalid mode"

##### 4. Revenue Analysis

In [182]:
# -------------------------------------------------
# 1. Inspect Target Revenue (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_rev_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "turnover"]
]

display(target_rev_df)


# -------------------------------------------------
# 2. Inspect Industry Revenue (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_rev_df = df[df["year"] == latest_year][
    ["company", "year", "turnover"]
]

display(industry_rev_df)

print("\nIndustry sample size:", len(industry_rev_df))


# -------------------------------------------------
# 3. Extract Inputs (Safe)
# -------------------------------------------------

if len(target_rev_df) == 0:
    rev_target_latest = np.nan
else:
    rev_target_latest = target_rev_df["turnover"].iloc[0]

rev_industry_latest = industry_rev_df["turnover"].mean()

print("\n--- CALCULATION INPUTS ---")
print(f"Target Revenue: {rev_target_latest:,.2f}" if pd.notna(rev_target_latest) else "Target Revenue: NaN")
print(f"Industry Mean Revenue: {rev_industry_latest:,.2f}")

# Avoid divide-by-zero
if pd.notna(rev_target_latest) and rev_industry_latest != 0:
    relative_pct = (rev_target_latest / rev_industry_latest - 1) * 100
else:
    relative_pct = np.nan

print("\n--- FORMULA ---")
print("Relative % = (Target / Industry Mean - 1) * 100")
print(f"= {relative_pct:.2f}%" if pd.notna(relative_pct) else "= NaN")


# -------------------------------------------------
# 4. Snapshot Result (Unified Engine)
# -------------------------------------------------

rev_snapshot = {
    "target": rev_target_latest,
    "industry_mean": rev_industry_latest,
    "relative_%": relative_pct,
    "classification": classify_metric(
        rev_target_latest,
        rev_industry_latest,
        mode="scale",
        scale_threshold=25
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([rev_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,turnover
19,Ekholms Bildemontering AB,2024,"3,447.00"



--- INDUSTRY DATA (Latest Year) ---


,company,year,turnover
3,Arboga Bilskrot AB,2024,"4,813.00"
7,Autocirc Svensk Bilåtervinning AB,2024,"41,596.00"
11,Bilåtervinning i Tibro AB,2024,"8,328.00"
15,Bjuhrs Bil AB,2024,"8,123.00"
19,Ekholms Bildemontering AB,2024,"3,447.00"
23,Köpings Bildemontering AB,2024,"7,825.00"
27,Mariestads Bildemontering och Återvinningstekn...,2024,"7,953.00"
31,Nya Högåsens Bildemontering AB,2024,"1,857.00"
35,Vingåkers Bilskrot AB,2024,"4,246.00"
39,Örebro Bildemontering AB,2024,"112,400.00"



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Revenue: 3,447.00
Industry Mean Revenue: 20,058.80

--- FORMULA ---
Relative % = (Target / Industry Mean - 1) * 100
= -82.82%

--- SNAPSHOT RESULT ---


,target,industry_mean,relative_%,classification
0,"3,447.00","20,058.80",-82.82,Subscale


##### 5. Profit Margin Analysis

In [183]:
# -------------------------------------------------
# 1. Inspect Target Data (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_pm_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "net_income", "turnover", "profit_margin"]
]

display(target_pm_df)


# -------------------------------------------------
# 2. Inspect Industry Data (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_pm_df = df[df["year"] == latest_year][
    ["company", "year", "net_income", "turnover", "profit_margin"]
]

display(industry_pm_df)

print("\nIndustry sample size:", len(industry_pm_df))


# -------------------------------------------------
# 3. Extract Inputs (Safe)
# -------------------------------------------------

if len(target_pm_df) == 0:
    pm_target_latest = np.nan
else:
    pm_target_latest = target_pm_df["profit_margin"].iloc[0]

pm_industry_latest = industry_pm_df["profit_margin"].mean()

print("\n--- CALCULATION INPUTS ---")
print(f"Target Profit Margin: {pm_target_latest:.2f}" if pd.notna(pm_target_latest) else "Target Profit Margin: NaN")
print(f"Industry Mean Profit Margin: {pm_industry_latest:.2f}")

# Safe gap calculation
if pd.notna(pm_target_latest):
    pm_gap = pm_target_latest - pm_industry_latest
else:
    pm_gap = np.nan

print("\n--- FORMULA ---")
print("Gap (pp) = Target - Industry Mean")
print(f"= {pm_gap:.2f} pp" if pd.notna(pm_gap) else "= NaN")


# -------------------------------------------------
# 4. Snapshot Result (Unified Engine)
# -------------------------------------------------

pm_snapshot = {
    "target": pm_target_latest,
    "industry_mean": pm_industry_latest,
    "gap_pp": pm_gap,
    "classification": classify_metric(
        pm_target_latest,
        pm_industry_latest,
        mode="absolute",
        threshold=5
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([pm_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,net_income,turnover,profit_margin
19,Ekholms Bildemontering AB,2024,NaN,"3,447.00",NaN



--- INDUSTRY DATA (Latest Year) ---


,company,year,net_income,turnover,profit_margin
3,Arboga Bilskrot AB,2024,78.00,"4,813.00",NaN
7,Autocirc Svensk Bilåtervinning AB,2024,NaN,"41,596.00",NaN
11,Bilåtervinning i Tibro AB,2024,716.00,"8,328.00",NaN
15,Bjuhrs Bil AB,2024,NaN,"8,123.00",NaN
19,Ekholms Bildemontering AB,2024,NaN,"3,447.00",NaN
23,Köpings Bildemontering AB,2024,NaN,"7,825.00",19.70
27,Mariestads Bildemontering och Återvinningstekn...,2024,426.00,"7,953.00",NaN
31,Nya Högåsens Bildemontering AB,2024,NaN,"1,857.00",NaN
35,Vingåkers Bilskrot AB,2024,713.00,"4,246.00",NaN
39,Örebro Bildemontering AB,2024,NaN,"112,400.00",NaN



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Profit Margin: NaN
Industry Mean Profit Margin: 19.70

--- FORMULA ---
Gap (pp) = Target - Industry Mean
= NaN

--- SNAPSHOT RESULT ---


,target,industry_mean,gap_pp,classification
0,NaN,19.70,NaN,Undefined


##### 6. Asset Turnover Analysis

In [152]:
# -------------------------------------------------
# 1. Inspect Target Selection
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_latest_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "turnover", "total_assets", "asset_turnover"]
]

display(target_latest_df)


# -------------------------------------------------
# 2. Inspect Industry Selection
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_latest_df = df[df["year"] == latest_year][
    ["company", "year", "turnover", "total_assets", "asset_turnover"]
]

display(industry_latest_df)

print("\nIndustry sample size:", len(industry_latest_df))


# -------------------------------------------------
# 3. Extract Calculation Inputs (Safe)
# -------------------------------------------------

if len(target_latest_df) == 0:
    at_target_latest = np.nan
else:
    at_target_latest = target_latest_df["asset_turnover"].iloc[0]

at_industry_latest = industry_latest_df["asset_turnover"].mean()

print("\n--- CALCULATION INPUTS ---")
print(f"Target Asset Turnover: {at_target_latest:.4f}" if pd.notna(at_target_latest) else "Target Asset Turnover: NaN")
print(f"Industry Mean Asset Turnover: {at_industry_latest:.4f}")

# Safe gap calculation
if pd.notna(at_target_latest):
    at_gap = at_target_latest - at_industry_latest
else:
    at_gap = np.nan

print("\n--- FORMULA ---")
print("Gap = Target - Industry Mean")
print(f"= {at_gap:.4f}" if pd.notna(at_gap) else "= NaN")


# -------------------------------------------------
# 4. Perform Snapshot Calculation (Unified Engine)
# -------------------------------------------------

at_snapshot = {
    "target": at_target_latest,
    "industry_mean": at_industry_latest,
    "gap": at_gap,
    "classification": classify_metric(
        at_target_latest,
        at_industry_latest,
        mode="ratio",       # ratio comparison
        threshold=0.2       # 0.2 turnover threshold
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([at_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,turnover,total_assets,asset_turnover
19,Ekholms Bildemontering AB,2024,"3,447.00","1,144.00",3.01



--- INDUSTRY DATA (Latest Year) ---


,company,year,turnover,total_assets,asset_turnover
3,Arboga Bilskrot AB,2024,"4,813.00","1,573.00",3.06
7,Autocirc Svensk Bilåtervinning AB,2024,"41,596.00","19,928.00",2.09
11,Bilåtervinning i Tibro AB,2024,"8,328.00","6,604.00",1.26
15,Bjuhrs Bil AB,2024,"8,123.00","4,599.00",1.77
19,Ekholms Bildemontering AB,2024,"3,447.00","1,144.00",3.01
23,Köpings Bildemontering AB,2024,"7,825.00","3,668.00",2.13
27,Mariestads Bildemontering och Återvinningstekn...,2024,"7,953.00","6,683.00",1.19
31,Nya Högåsens Bildemontering AB,2024,"1,857.00","2,439.00",0.76
35,Vingåkers Bilskrot AB,2024,"4,246.00","2,352.00",1.81
39,Örebro Bildemontering AB,2024,"112,400.00","67,269.00",1.67



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Asset Turnover: 3.0131
Industry Mean Asset Turnover: 1.8748

--- FORMULA ---
Gap = Target - Industry Mean
= 1.1383

--- SNAPSHOT RESULT ---


,target,industry_mean,gap,classification
0,3.01,1.87,1.14,Strong


##### 7. Return on Equity (ROE) Analysis

In [153]:
# -------------------------------------------------
# 1. Inspect Target Selection
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_roe_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(target_roe_df)


# -------------------------------------------------
# 2. Inspect Industry Selection
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_roe_df = df[df["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(industry_roe_df)

print("\nIndustry sample size:", len(industry_roe_df))


# -------------------------------------------------
# 3. Extract Calculation Inputs (Safe)
# -------------------------------------------------

if len(target_roe_df) == 0:
    roe_target_latest = np.nan
else:
    roe_target_latest = target_roe_df["return_on_equity"].iloc[0]

roe_industry_latest = industry_roe_df["return_on_equity"].mean()

print("\n--- CALCULATION INPUTS ---")
print(f"Target ROE: {roe_target_latest:.2f}" if pd.notna(roe_target_latest) else "Target ROE: NaN")
print(f"Industry Mean ROE: {roe_industry_latest:.2f}")

# Safe gap calculation
if pd.notna(roe_target_latest):
    roe_gap = roe_target_latest - roe_industry_latest
else:
    roe_gap = np.nan

print("\n--- FORMULA ---")
print("Gap (pp) = Target - Industry Mean")
print(f"= {roe_gap:.2f} percentage points" if pd.notna(roe_gap) else "= NaN")


# -------------------------------------------------
# 4. Perform Snapshot Calculation (Unified Engine)
# -------------------------------------------------

roe_snapshot = {
    "target": roe_target_latest,
    "industry_mean": roe_industry_latest,
    "gap_pp": roe_gap,
    "classification": classify_metric(
        roe_target_latest,
        roe_industry_latest,
        mode="absolute",
        threshold=5
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([roe_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,return_on_equity
19,Ekholms Bildemontering AB,2024,NaN



--- INDUSTRY DATA (Latest Year) ---


,company,year,return_on_equity
3,Arboga Bilskrot AB,2024,NaN
7,Autocirc Svensk Bilåtervinning AB,2024,NaN
11,Bilåtervinning i Tibro AB,2024,NaN
15,Bjuhrs Bil AB,2024,NaN
19,Ekholms Bildemontering AB,2024,NaN
23,Köpings Bildemontering AB,2024,73.00
27,Mariestads Bildemontering och Återvinningstekn...,2024,NaN
31,Nya Högåsens Bildemontering AB,2024,NaN
35,Vingåkers Bilskrot AB,2024,NaN
39,Örebro Bildemontering AB,2024,NaN



Industry sample size: 10

--- CALCULATION INPUTS ---
Target ROE: NaN
Industry Mean ROE: 73.00

--- FORMULA ---
Gap (pp) = Target - Industry Mean
= NaN

--- SNAPSHOT RESULT ---


,target,industry_mean,gap_pp,classification
0,NaN,73.00,NaN,Undefined


##### 8. Equity Ratio Analysis

In [154]:
# -------------------------------------------------
# 1. Inspect Target Selection
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_roe_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(target_roe_df)


# -------------------------------------------------
# 2. Inspect Industry Selection
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_roe_df = df[df["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(industry_roe_df)

print("\nIndustry sample size:", len(industry_roe_df))


# -------------------------------------------------
# 3. Extract Calculation Inputs (Safe)
# -------------------------------------------------

if len(target_roe_df) == 0:
    roe_target_latest = np.nan
else:
    roe_target_latest = target_roe_df["return_on_equity"].iloc[0]

roe_industry_latest = industry_roe_df["return_on_equity"].mean()

print("\n--- CALCULATION INPUTS ---")
print(f"Target ROE: {roe_target_latest:.2f}" if pd.notna(roe_target_latest) else "Target ROE: NaN")
print(f"Industry Mean ROE: {roe_industry_latest:.2f}")

if pd.notna(roe_target_latest):
    roe_gap = roe_target_latest - roe_industry_latest
else:
    roe_gap = np.nan

print("\n--- FORMULA ---")
print("Gap (pp) = Target - Industry Mean")
print(f"= {roe_gap:.2f} percentage points" if pd.notna(roe_gap) else "= NaN")


# -------------------------------------------------
# 4. Snapshot Result
# -------------------------------------------------

roe_snapshot = {
    "target": roe_target_latest,
    "industry_mean": roe_industry_latest,
    "gap_pp": roe_gap,
    "classification": classify_metric(
        roe_target_latest,
        roe_industry_latest,
        mode="absolute",
        threshold=5
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([roe_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,return_on_equity
19,Ekholms Bildemontering AB,2024,NaN



--- INDUSTRY DATA (Latest Year) ---


,company,year,return_on_equity
3,Arboga Bilskrot AB,2024,NaN
7,Autocirc Svensk Bilåtervinning AB,2024,NaN
11,Bilåtervinning i Tibro AB,2024,NaN
15,Bjuhrs Bil AB,2024,NaN
19,Ekholms Bildemontering AB,2024,NaN
23,Köpings Bildemontering AB,2024,73.00
27,Mariestads Bildemontering och Återvinningstekn...,2024,NaN
31,Nya Högåsens Bildemontering AB,2024,NaN
35,Vingåkers Bilskrot AB,2024,NaN
39,Örebro Bildemontering AB,2024,NaN



Industry sample size: 10

--- CALCULATION INPUTS ---
Target ROE: NaN
Industry Mean ROE: 73.00

--- FORMULA ---
Gap (pp) = Target - Industry Mean
= NaN

--- SNAPSHOT RESULT ---


,target,industry_mean,gap_pp,classification
0,NaN,73.00,NaN,Undefined


##### 9. Revenue Structure

In [145]:
# -------------------------------------------------
# 1. Inspect Target Time Series
# -------------------------------------------------

print("\n--- TARGET REVENUE TIME SERIES ---")

rev_trend_df = (
    df_target[["year", "turnover"]]
    .sort_values("year")
)

display(rev_trend_df)

print("\nNumber of observations:", len(rev_trend_df))


# -------------------------------------------------
# 2. Compute Derived Series
# -------------------------------------------------

rev_trend = rev_trend_df.set_index("year")["turnover"]

rev_growth = rev_trend.pct_change()

display(pd.DataFrame({
    "Revenue": rev_trend,
    "YoY Growth": rev_growth
}))

# -------------------------------------------------
# 3. Extract Calculation Inputs
# -------------------------------------------------

rev_cagr = (
    (rev_trend.iloc[-1] / rev_trend.iloc[0]) **
    (1 / (len(rev_trend) - 1))
    - 1
)

rev_volatility = rev_growth.std()

print("\n--- CALCULATION INPUTS ---")
print(f"Start Revenue: {rev_trend.iloc[0]:,.2f}")
print(f"End Revenue: {rev_trend.iloc[-1]:,.2f}")
print(f"Periods: {len(rev_trend) - 1}")

print("\n--- FORMULA ---")
print(
    f"CAGR = ({rev_trend.iloc[-1]:,.2f} / "
    f"{rev_trend.iloc[0]:,.2f})^(1/{len(rev_trend)-1}) - 1"
)

print(f"= {rev_cagr:.4f}")

print("\nVolatility (std of YoY growth):")
print(f"= {rev_volatility:.4f}")


# -------------------------------------------------
# 4. Structured Snapshot Output
# -------------------------------------------------

# -------------------------------------------------
# 4. Revenue Structure Snapshot Result
# -------------------------------------------------

rev_structure_snapshot = {
    "initial_year": rev_trend.index.min(),
    "final_year": rev_trend.index.max(),
    "initial_revenue": rev_trend.iloc[0],
    "final_revenue": rev_trend.iloc[-1],
    "CAGR_%": round(rev_cagr * 100, 2) if pd.notna(rev_cagr) else np.nan,
    "Volatility_%": round(rev_volatility * 100, 2) if pd.notna(rev_volatility) else np.nan,
    "observations": len(rev_trend)
}

print("\n--- REVENUE STRUCTURE SNAPSHOT ---")
display(pd.DataFrame([rev_structure_snapshot]))


--- TARGET REVENUE TIME SERIES ---


,year,turnover
16,2021,"6,428.00"
17,2022,"6,519.00"
18,2023,"4,206.00"
19,2024,"3,447.00"



Number of observations: 4


,Revenue,YoY Growth
year,,
2021,"6,428.00",NaN
2022,"6,519.00",0.01
2023,"4,206.00",-0.35
2024,"3,447.00",-0.18



--- CALCULATION INPUTS ---
Start Revenue: 6,428.00
End Revenue: 3,447.00
Periods: 3

--- FORMULA ---
CAGR = (3,447.00 / 6,428.00)^(1/3) - 1
= -0.1876

Volatility (std of YoY growth):
= 0.1846

--- REVENUE STRUCTURE SNAPSHOT ---


,initial_year,final_year,initial_revenue,final_revenue,CAGR_%,Volatility_%,observations
0,2021,2024,"6,428.00","3,447.00",-18.76,18.46,4


##### 10. Margin Structure

In [146]:
# -------------------------------------------------
# 1. Inspect Target Margin Time Series
# -------------------------------------------------

print("\n--- TARGET PROFIT MARGIN TIME SERIES ---")

pm_trend_df = df_target[["year", "profit_margin"]].sort_values("year")

display(pm_trend_df)

print("\nObservation count:", len(pm_trend_df))


# -------------------------------------------------
# 2. Extract Series (Clean NaNs)
# -------------------------------------------------

pm_trend = pm_trend_df.set_index("year")["profit_margin"]

print("\n--- RAW SERIES ---")
print(pm_trend)

pm_trend_clean = pm_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(pm_trend_clean)

print("Valid observations:", len(pm_trend_clean))


# -------------------------------------------------
# 3. Perform Calculations (Safe)
# -------------------------------------------------

if len(pm_trend_clean) > 0:
    pm_mean = pm_trend_clean.mean()
    pm_std = pm_trend_clean.std()
    pm_min = pm_trend_clean.min()
    pm_max = pm_trend_clean.max()
else:
    pm_mean = pm_std = pm_min = pm_max = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average margin")
print("StdDev = Margin volatility")
print("Min = Worst year")
print("Max = Best year")


# -------------------------------------------------
# 4. Margin Structure Snapshot
# -------------------------------------------------

pm_structure_snapshot = {
    "initial_year": pm_trend.index.min(),
    "final_year": pm_trend.index.max(),
    "Mean_%": round(pm_mean, 2) if pd.notna(pm_mean) else np.nan,
    "StdDev_%": round(pm_std, 2) if pd.notna(pm_std) else np.nan,
    "Min_%": round(pm_min, 2) if pd.notna(pm_min) else np.nan,
    "Max_%": round(pm_max, 2) if pd.notna(pm_max) else np.nan,
    "valid_observations": len(pm_trend_clean)
}

print("\n--- MARGIN STRUCTURE SNAPSHOT ---")
display(pd.DataFrame([pm_structure_snapshot]))


--- TARGET PROFIT MARGIN TIME SERIES ---


,year,profit_margin
16,2021,NaN
17,2022,NaN
18,2023,NaN
19,2024,NaN



Observation count: 4

--- RAW SERIES ---
year
2021   NaN
2022   NaN
2023   NaN
2024   NaN
Name: profit_margin, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
Series([], Name: profit_margin, dtype: float64)
Valid observations: 0

--- FORMULAS ---
Mean = Average margin
StdDev = Margin volatility
Min = Worst year
Max = Best year

--- MARGIN STRUCTURE SNAPSHOT ---


,initial_year,final_year,Mean_%,StdDev_%,Min_%,Max_%,valid_observations
0,2021,2024,NaN,NaN,NaN,NaN,0


##### 11. Asset Turnover Trend

In [147]:
# -------------------------------------------------
# 1. Inspect Target Asset Turnover Time Series
# -------------------------------------------------

print("\n--- TARGET ASSET TURNOVER TIME SERIES ---")

at_trend_df = df_target[["year", "asset_turnover"]].sort_values("year")

display(at_trend_df)

print("\nObservation count:", len(at_trend_df))


# -------------------------------------------------
# 2. Extract Series (Clean NaNs)
# -------------------------------------------------

at_trend = at_trend_df.set_index("year")["asset_turnover"]

print("\n--- RAW SERIES ---")
print(at_trend)

at_trend_clean = at_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(at_trend_clean)

print("Valid observations:", len(at_trend_clean))


# -------------------------------------------------
# 3. Perform Calculations (Safe)
# -------------------------------------------------

if len(at_trend_clean) > 0:
    at_mean = at_trend_clean.mean()
    at_std = at_trend_clean.std()

    # Simple directional change (first vs last)
    at_direction = at_trend_clean.iloc[-1] - at_trend_clean.iloc[0]

    # Optional: structural slope (more robust)
    if len(at_trend_clean) > 1:
        years = at_trend_clean.index.values
        values = at_trend_clean.values
        slope = np.polyfit(years, values, 1)[0]
    else:
        slope = np.nan

else:
    at_mean = at_std = at_direction = slope = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average asset turnover")
print("StdDev = Volatility")
print("Direction = Last - First")
print("Slope = Linear regression slope (structural drift)")


# -------------------------------------------------
# 4. Asset Turnover Trend Snapshot
# -------------------------------------------------

at_trend_snapshot = {
    "initial_year": at_trend.index.min(),
    "final_year": at_trend.index.max(),
    "Mean": round(at_mean, 4) if pd.notna(at_mean) else np.nan,
    "StdDev": round(at_std, 4) if pd.notna(at_std) else np.nan,
    "Direction": round(at_direction, 4) if pd.notna(at_direction) else np.nan,
    "Structural_Slope": round(slope, 4) if pd.notna(slope) else np.nan,
    "valid_observations": len(at_trend_clean)
}

print("\n--- ASSET TURNOVER TREND SNAPSHOT ---")
display(pd.DataFrame([at_trend_snapshot]))


--- TARGET ASSET TURNOVER TIME SERIES ---


,year,asset_turnover
16,2021,2.84
17,2022,2.60
18,2023,2.96
19,2024,3.01



Observation count: 4

--- RAW SERIES ---
year
2021   2.84
2022   2.60
2023   2.96
2024   3.01
Name: asset_turnover, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
year
2021   2.84
2022   2.60
2023   2.96
2024   3.01
Name: asset_turnover, dtype: float64
Valid observations: 4

--- FORMULAS ---
Mean = Average asset turnover
StdDev = Volatility
Direction = Last - First
Slope = Linear regression slope (structural drift)

--- ASSET TURNOVER TREND SNAPSHOT ---


,initial_year,final_year,Mean,StdDev,Direction,Structural_Slope,valid_observations
0,2021,2024,2.85,0.18,0.18,0.09,4


##### 12. ROE Trend

In [148]:
# -------------------------------------------------
# 1. Inspect Target ROE Time Series
# -------------------------------------------------

print("\n--- TARGET ROE TIME SERIES ---")

roe_trend_df = df_target[["year", "return_on_equity"]].sort_values("year")

display(roe_trend_df)

print("\nObservation count:", len(roe_trend_df))


# -------------------------------------------------
# 2. Extract Series (Clean NaNs)
# -------------------------------------------------

roe_trend = roe_trend_df.set_index("year")["return_on_equity"]

print("\n--- RAW SERIES ---")
print(roe_trend)

roe_trend_clean = roe_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(roe_trend_clean)

print("Valid observations:", len(roe_trend_clean))


# -------------------------------------------------
# 3. Perform Calculations (Safe)
# -------------------------------------------------

if len(roe_trend_clean) > 0:

    roe_mean = roe_trend_clean.mean()
    roe_std = roe_trend_clean.std()

    # Endpoint directional change
    roe_direction = roe_trend_clean.iloc[-1] - roe_trend_clean.iloc[0]

    # Structural regression slope (more robust)
    if len(roe_trend_clean) > 1:
        years = roe_trend_clean.index.values
        values = roe_trend_clean.values
        roe_slope = np.polyfit(years, values, 1)[0]
    else:
        roe_slope = np.nan

else:
    roe_mean = roe_std = roe_direction = roe_slope = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average ROE")
print("StdDev = ROE volatility")
print("Direction = Last - First")
print("Slope = Linear regression slope (structural drift)")


# -------------------------------------------------
# 4. ROE Trend Snapshot
# -------------------------------------------------

roe_trend_snapshot = {
    "initial_year": roe_trend.index.min(),
    "final_year": roe_trend.index.max(),
    "Mean_%": round(roe_mean, 2) if pd.notna(roe_mean) else np.nan,
    "StdDev_%": round(roe_std, 2) if pd.notna(roe_std) else np.nan,
    "Direction_pp": round(roe_direction, 2) if pd.notna(roe_direction) else np.nan,
    "Structural_Slope": round(roe_slope, 4) if pd.notna(roe_slope) else np.nan,
    "valid_observations": len(roe_trend_clean)

}

print("\n--- ROE TREND SNAPSHOT ---")
display(pd.DataFrame([roe_trend_snapshot]))


--- TARGET ROE TIME SERIES ---


,year,return_on_equity
16,2021,NaN
17,2022,NaN
18,2023,NaN
19,2024,NaN



Observation count: 4

--- RAW SERIES ---
year
2021   NaN
2022   NaN
2023   NaN
2024   NaN
Name: return_on_equity, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
Series([], Name: return_on_equity, dtype: float64)
Valid observations: 0

--- FORMULAS ---
Mean = Average ROE
StdDev = ROE volatility
Direction = Last - First
Slope = Linear regression slope (structural drift)

--- ROE TREND SNAPSHOT ---


,initial_year,final_year,Mean_%,StdDev_%,Direction_pp,Structural_Slope,valid_observations
0,2021,2024,NaN,NaN,NaN,NaN,0


##### 13. Equity Ratio Trend

In [149]:
# -------------------------------------------------
# 1. Inspect Target Equity Ratio Time Series
# -------------------------------------------------

print("\n--- TARGET EQUITY RATIO TIME SERIES ---")

eq_trend_df = df_target[["year", "equity_ratio"]].sort_values("year")

display(eq_trend_df)

print("\nObservation count:", len(eq_trend_df))


# -------------------------------------------------
# 2. Extract Series (Clean NaNs)
# -------------------------------------------------

eq_trend = eq_trend_df.set_index("year")["equity_ratio"]

print("\n--- RAW SERIES ---")
print(eq_trend)

eq_trend_clean = eq_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(eq_trend_clean)

print("Valid observations:", len(eq_trend_clean))


# -------------------------------------------------
# 3. Perform Calculations (Safe)
# -------------------------------------------------

if len(eq_trend_clean) > 0:

    eq_mean = eq_trend_clean.mean()
    eq_std = eq_trend_clean.std()

    # Endpoint directional change
    eq_direction = eq_trend_clean.iloc[-1] - eq_trend_clean.iloc[0]

    # Structural regression slope
    if len(eq_trend_clean) > 1:
        years = eq_trend_clean.index.values
        values = eq_trend_clean.values
        eq_slope = np.polyfit(years, values, 1)[0]
    else:
        eq_slope = np.nan

else:
    eq_mean = eq_std = eq_direction = eq_slope = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average equity ratio")
print("StdDev = Volatility")
print("Direction = Last - First")
print("Slope = Linear regression slope (structural drift)")


# -------------------------------------------------
# 4. Equity Ratio Trend Snapshot
# -------------------------------------------------

eq_trend_snapshot = {
    "initial_year": eq_trend.index.min(),
    "final_year": eq_trend.index.max(),
    "Mean": round(eq_mean, 4) if pd.notna(eq_mean) else np.nan,
    "StdDev": round(eq_std, 4) if pd.notna(eq_std) else np.nan,
    "Direction": round(eq_direction, 4) if pd.notna(eq_direction) else np.nan,
    "Structural_Slope": round(eq_slope, 6) if pd.notna(eq_slope) else np.nan,
    "valid_observations": len(eq_trend_clean)
}

print("\n--- EQUITY RATIO TREND SNAPSHOT ---")
display(pd.DataFrame([eq_trend_snapshot]))


--- TARGET EQUITY RATIO TIME SERIES ---


KeyError: "['equity_ratio'] not in index"

##### 14. Consolidate Snapshot

In [87]:
financial_snapshot = {
    "revenue": rev_snapshot,
    "profit_margin": pm_snapshot,
    "asset_turnover": at_snapshot,
    "roe": roe_snapshot,
    "equity_ratio": eq_snapshot
}

financial_snapshot

NameError: name 'eq_snapshot' is not defined